In [2]:
import pandas as pd
import numpy as np

# Load the raw dataset
# We use 'df' (short for DataFrame), which is like an Excel table in Python memory
df = pd.read_csv('Maritime Project AI.csv')

# Show the first 5 rows to understand the "Messy" state
print(f"Dataset Dimensions: {df.shape}")
df.head()

Dataset Dimensions: (803, 20)


,Unnamed: 0,Economy_Label,CommercialMarket_Label,Average_age_of_vessels_years_Value,Average_age_of_vessels_years_MissingValue,Median_time_in_port_days_Value,Median_time_in_port_days_MissingValue,Average_size_GT_of_vessels_Value,Average_size_GT_of_vessels_MissingValue,Average_cargo_carrying_capacity_dwt_per_vessel_Value,Average_cargo_carrying_capacity_dwt_per_vessel_MissingValue,Average_container_carrying_capacity_TEU_per_container_ship_Value,Average_container_carrying_capacity_TEU_per_container_ship_MissingValue,Maximum_size_GT_of_vessels_Value,Maximum_size_GT_of_vessels_MissingValue,Maximum_cargo_carrying_capacity_dwt_of_vessels_Value,Maximum_cargo_carrying_capacity_dwt_of_vessels_MissingValue,Maximum_container_carrying_capacity_TEU_of_container_ships_Value,Maximum_container_carrying_capacity_TEU_of_container_ships_MissingValue,period
0,0,World,All ships,18,NaN,1.07,NaN,14285,NaN,25725.0,NaN,3407.0,NaN,236583,NaN,404389.0,NaN,24000.0,NaN,2022-S1
1,1,World,Liquid bulk carriers,15,NaN,1.00,NaN,16298,NaN,28286.0,NaN,NaN,Not available or not separately reported,170618,NaN,323183.0,NaN,NaN,Not available or not separately reported,2022-S1
2,2,World,Liquefied petroleum gas carriers,16,NaN,1.03,NaN,10726,NaN,11986.0,NaN,NaN,Not available or not separately reported,60784,NaN,64220.0,NaN,NaN,Not available or not separately reported,2022-S1
3,3,World,Liquefied natural gas carriers,12,NaN,1.12,NaN,96843,NaN,75614.0,NaN,NaN,Not available or not separately reported,168189,NaN,155159.0,NaN,NaN,Not available or not separately reported,2022-S1
4,4,World,Dry bulk carriers,14,NaN,2.23,NaN,32735,NaN,58640.0,NaN,NaN,Not available or not separately reported,204014,NaN,404389.0,NaN,NaN,Not available or not separately reported,2022-S1


In [3]:
# Remove columns that don't provide analytical value
# We drop 'Unnamed: 0' (old index) and any column ending in '_MissingValue'
cols_to_drop = [col for col in df.columns if col.endswith('_MissingValue') or col == 'Unnamed: 0']
df_clean = df.drop(columns=cols_to_drop)

print(f"Columns remaining after pruning: {len(df_clean.columns)}")

Columns remaining after pruning: 11


In [4]:
# Mapping old names to short, professional headers
rename_map = {
    'Economy_Label': 'Region',
    'CommercialMarket_Label': 'Ship_Type',
    'Average_age_of_vessels_years_Value': 'Avg_Age',
    'Median_time_in_port_days_Value': 'Median_Port_Time',
    'Average_size_GT_of_vessels_Value': 'Avg_Size_GT',
    'Average_cargo_carrying_capacity_dwt_per_vessel_Value': 'Avg_Cargo_DWT',
    'Average_container_carrying_capacity_TEU_per_container_ship_Value': 'Avg_TEU',
    'Maximum_size_GT_of_vessels_Value': 'Max_Size_GT',
    'Maximum_cargo_carrying_capacity_dwt_of_vessels_Value': 'Max_Cargo_DWT',
    'Maximum_container_carrying_capacity_TEU_of_container_ships_Value': 'Max_TEU',
    'period': 'Period_Code'
}

df_clean.rename(columns=rename_map, inplace=True)
print("Columns renamed for standardization.")

Columns renamed for standardization.


In [5]:
# Create new columns by splitting the Period_Code string
df_clean['Year'] = df_clean['Period_Code'].str.split('-S').str[0].astype(int)
df_clean['Semester'] = df_clean['Period_Code'].str.split('-S').str[1].astype(int)

# Check our new columns
df_clean[['Period_Code', 'Year', 'Semester']].head()

,Period_Code,Year,Semester
0,2022-S1,2022,1
1,2022-S1,2022,1
2,2022-S1,2022,1
3,2022-S1,2022,1
4,2022-S1,2022,1


In [6]:
# Fill container and cargo capacities with 0 if they are blank (Logic-based imputation)
fill_zero_cols = ['Avg_TEU', 'Max_TEU', 'Avg_Cargo_DWT', 'Max_Cargo_DWT']
df_clean[fill_zero_cols] = df_clean[fill_zero_cols].fillna(0)

# Fill Port Time with the median (Statistical imputation)
# This prevents skewed results from outliers
df_clean['Median_Port_Time'] = df_clean['Median_Port_Time'].fillna(df_clean['Median_Port_Time'].median())

print("Missing values handled.")

Missing values handled.


In [7]:
# Save the final product
df_clean.to_csv('Cleaned_Maritime_Data.csv', index=False)

print("ETL SUCCESS: Cleaned_Maritime_Data.csv is ready.")
df_clean.info() # Show final summary of the data

ETL SUCCESS: Cleaned_Maritime_Data.csv is ready.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 803 entries, 0 to 802
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Region            803 non-null    object 
 1   Ship_Type         803 non-null    object 
 2   Avg_Age           803 non-null    int64  
 3   Median_Port_Time  803 non-null    float64
 4   Avg_Size_GT       803 non-null    int64  
 5   Avg_Cargo_DWT     803 non-null    float64
 6   Avg_TEU           803 non-null    float64
 7   Max_Size_GT       803 non-null    int64  
 8   Max_Cargo_DWT     803 non-null    float64
 9   Max_TEU           803 non-null    float64
 10  Period_Code       803 non-null    object 
 11  Year              803 non-null    int64  
 12  Semester          803 non-null    int64  
dtypes: float64(5), int64(5), object(3)
memory usage: 81.7+ KB
